## Download the source data file from kaggle

In [ ]:
import kagglehub
path = kagglehub.dataset_download("tiagoadrianunes/imdb-top-5000-movies")

Using Colab cache for faster access to the 'imdb-top-5000-movies' dataset.


## Import necessary libraries

In [ ]:
import os
from pyspark.sql import SparkSession

## Extract data from file with PySpark

In [ ]:
spark= SparkSession.builder.getOrCreate()
movies_df=spark.read.csv(os.path.join(path, "results_with_crew.csv"), header=True, inferSchema=True)

In [ ]:
movies_df.show(2)

+---------+--------------------+---------+----+-------------+--------+--------------+--------------------+--------------------+------------+--------------------+--------------------+
|   tconst|        primaryTitle|startYear|rank|averageRating|numVotes|runtimeMinutes|           directors|             writers|      genres|            IMDbLink|     Title_IMDb_Link|
+---------+--------------------+---------+----+-------------+--------+--------------+--------------------+--------------------+------------+--------------------+--------------------+
|tt0111161|The Shawshank Red...|     1994|   1|          9.3| 3246333|           142|      Frank Darabont|Stephen King, Fra...|       Drama|"<a href=""https:...|"<a href=""https:...|
|tt0068646|       The Godfather|     1972|   2|          9.2| 2261044|           175|Francis Ford Coppola|Mario Puzo, Franc...|Crime, Drama|"<a href=""https:...|"<a href=""https:...|
+---------+--------------------+---------+----+-------------+--------+--------------+

In [ ]:
movies_df.columns

['tconst',
 'primaryTitle',
 'startYear',
 'rank',
 'averageRating',
 'numVotes',
 'runtimeMinutes',
 'directors',
 'writers',
 'genres',
 'IMDbLink',
 'Title_IMDb_Link']

## Mount Google Drive

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
DB_PATH= "/content/drive/MyDrive/data_engineering/movies.duckdb"

### Saving extracted data into bronze tables

In [24]:
import duckdb

In [26]:
con= duckdb.connect(DB_PATH)

In [27]:
con.register(f"movies_bronze_tmp",movies_df.toPandas())
con.execute("""
  CREATE OR REPLACE TABLE movies_bronze AS
  SELECT * FROM movies_bronze_tmp
""")


In [28]:
con.unregister("movies_bronze_tmp")

In [29]:
con.execute("""
  SELECT * FROM movies_bronze LIMIT 2
""").fetchdf()

,tconst,primaryTitle,startYear,rank,averageRating,numVotes,runtimeMinutes,directors,writers,genres,IMDbLink,Title_IMDb_Link
0,tt0111161,The Shawshank Redemption,1994,1,9.3,3246333,142,Frank Darabont,"Stephen King, Frank Darabont",Drama,"""<a href=""""https://www.imdb.com/title/tt011116...","""<a href=""""https://www.imdb.com/title/tt011116..."
1,tt0068646,The Godfather,1972,2,9.2,2261044,175,Francis Ford Coppola,"Mario Puzo, Francis Ford Coppola","Crime, Drama","""<a href=""""https://www.imdb.com/title/tt006864...","""<a href=""""https://www.imdb.com/title/tt006864..."
